In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w8" / "day01"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [2]:
import json

DRAFT_SCHEMA = {'name': 'search_documents', 'description': '사내 규정 문서에서 질문과 관련된 근거를 찾는다. 출장비·여비·숙박비·법인카드처럼 규정에 적힌 값을 물어보면 이 도구를 쓴다.', 'input_schema': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': '찾을 내용. 사용자의 질문을 그대로 넣어도 된다'}}, 'required': ['query']}}

print(json.dumps(DRAFT_SCHEMA, ensure_ascii=False, indent=2))
print()
print('바깥 칸       :', list(DRAFT_SCHEMA.keys()))
print('input_schema 칸 :', list(DRAFT_SCHEMA['input_schema'].keys()))

{
  "name": "search_documents",
  "description": "사내 규정 문서에서 질문과 관련된 근거를 찾는다. 출장비·여비·숙박비·법인카드처럼 규정에 적힌 값을 물어보면 이 도구를 쓴다.",
  "input_schema": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "찾을 내용. 사용자의 질문을 그대로 넣어도 된다"
      }
    },
    "required": [
      "query"
    ]
  }
}

바깥 칸       : ['name', 'description', 'input_schema']
input_schema 칸 : ['type', 'properties', 'required']


In [3]:
from anthropic.types import Message
FAKE_RESP = {'id': 'msg_01FAKE', 'type': 'message', 'role': 'assistant', 'model': 'claude-haiku-4-5', 'content': [{'type': 'text', 'text': '사내 규정을 찾아보겠습니다.'}, {'type': 'tool_use', 'id': 'toolu_01ABC', 'name': 'search_documents', 'input': {'query': '부산 출장 숙박비'}}], 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'input_tokens': 412, 'output_tokens': 57}}
resp = Message.model_validate(FAKE_RESP)
print('stop_reason      =', repr(resp.stop_reason))
print('content types    =', [b.type for b in resp.content])
print('block classes    =', [type(b).__name__ for b in resp.content])
print()
text = ''.join((b.text for b in resp.content if getattr(b, 'type', '') == 'text'))
print('text 만 주우면   =', repr(text))
use = [b for b in resp.content if getattr(b, 'type', '') == 'tool_use'][0]
print('tool_use 블록    :', use.id, '/', use.name, '/', use.input)
print('input 의 형       :', type(use.input))


stop_reason      = 'tool_use'
content types    = ['text', 'tool_use']
block classes    = ['TextBlock', 'ToolUseBlock']

text 만 주우면   = '사내 규정을 찾아보겠습니다.'
tool_use 블록    : toolu_01ABC / search_documents / {'query': '부산 출장 숙박비'}
input 의 형       : <class 'dict'>


In [4]:
TOOL_OUT = {'sources': [{'title': '국내출장 여비 규정', 'version': 'v2.0', 'locator': '제12조(숙박비)'}], 'passed': True, 'best_score': 0.89}

messages = [{'role': 'user', 'content': '부산 출장 숙박비 얼마까지 되나요?'}]
messages.append({'role': 'assistant', 'content': [b.model_dump(exclude_none=True) for b in resp.content]})
messages.append({'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': use.id, 'content': json.dumps(TOOL_OUT, ensure_ascii=False)}]})
print(json.dumps(messages, ensure_ascii=False, indent=2))
print()

print('messages 덩어리 :', len(messages))
print('json.dumps(...)                     길이', len(json.dumps(TOOL_OUT)))
print('json.dumps(..., ensure_ascii=False) 길이', len(json.dumps(TOOL_OUT, ensure_ascii=False)))


[
  {
    "role": "user",
    "content": "부산 출장 숙박비 얼마까지 되나요?"
  },
  {
    "role": "assistant",
    "content": [
      {
        "text": "사내 규정을 찾아보겠습니다.",
        "type": "text"
      },
      {
        "id": "toolu_01ABC",
        "input": {
          "query": "부산 출장 숙박비"
        },
        "name": "search_documents",
        "type": "tool_use"
      }
    ]
  },
  {
    "role": "user",
    "content": [
      {
        "type": "tool_result",
        "tool_use_id": "toolu_01ABC",
        "content": "{\"sources\": [{\"title\": \"국내출장 여비 규정\", \"version\": \"v2.0\", \"locator\": \"제12조(숙박비)\"}], \"passed\": true, \"best_score\": 0.89}"
      }
    ]
  }
]

messages 덩어리 : 3
json.dumps(...)                     길이 182
json.dumps(..., ensure_ascii=False) 길이 117


In [5]:
from app.core.config import get_settings
from app.integrations import factory
from app.integrations.ports import LLMResult, ToolCall
from app.services import chat_service
settings = get_settings()
HAS_KEY = settings.is_live and settings.anthropic_api_key is not None
QUESTIONS = ['부산 출장 숙박비 얼마까지 되나요?', '법인카드 한도가 얼마인가요?']

class _CannedLLM:
    """키가 없을 때 factory.get_llm 자리에 잠깐 끼우는 대역. 네트워크를 타지 않는다.

    미리 뽑아 둔 응답은 질문 하나에서 뽑은 것이라 검색어가 "부산 출장 숙박비" 로
    고정돼 있다. 오늘은 질문이 둘이라 **질문을 그대로 검색어로 넣는다** —
    키 없는 길에서는 「모델이 검색어를 정하는」 장면을 볼 수 없다
    """

    def __init__(self, question: str) -> None:
        self.question, self.turn = (question, 0)

    def answer_with_tools(self, *, messages, tools, user) -> LLMResult:
        self.turn += 1
        if self.turn == 1:
            call = ToolCall(id='toolu_01ABC', name='search_documents', args={'query': self.question})
            return LLMResult(text='사내 규정을 찾아보겠습니다.', model=settings.llm_model, input_tok=412, output_tok=57, extras={'tool_calls': [call], 'stop_reason': 'tool_use', 'blocks': [{'type': 'tool_use', 'id': call.id, 'name': call.name, 'input': call.args}]})
        return LLMResult(text='(미리 뽑아 둔 답변 본문)', model=settings.llm_model, input_tok=690, output_tok=31, extras={'tool_calls': [], 'stop_reason': 'end_turn', 'blocks': []})
print('길 :', '진짜 호출' if HAS_KEY else '미리 뽑아 둔 값')
for i, q in enumerate(QUESTIONS, 1):
    original = factory.get_llm
    if not HAS_KEY:
        factory.get_llm = lambda _stub=_CannedLLM(q): _stub
    try:
        out = chat_service.ask_with_tools(question=q)
    finally:
        factory.get_llm = original
    print(f'[{i}] {q}')
    print(f'    run_id {out.run_id} · 근거 {len(out.sources)}건 · 답변 {out.answer[:40]}')
    print(f'    단계 {[st['name'] for st in out.steps]}')

길 : 진짜 호출
14:12:13 INFO     httpx2: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
14:12:29 INFO     sentence_transformers.base.model: No device provided, using cpu
14:12:29 INFO     sentence_transformers.base.model: Loading SentenceTransformer model from C:\hw\hw\python_basic\models\bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

14:12:32 INFO     app.integrations.local_embed: 로컬 임베딩 모델 준비: C:\hw\hw\python_basic\models\bge-m3 / 1024차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

14:12:37 INFO     app.integrations.local_rerank: 로컬 리랭커 준비 : ..\models\bge-reranker-v2-m3, 최대 512토큰
14:12:50 INFO     httpx2: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
[1] 부산 출장 숙박비 얼마까지 되나요?
    run_id RUN-8861 · 근거 3건 · 답변 좋은 질문입니다! 회사의 **국내출장 여비 규정**에 따르면 부산 출장 
    단계 ['질의 재작성 · Multi-Query 2건 생성', '벡터 + 키워드 하이브리드 검색 · 벡터 16건 · 키워드 12건 -> 후보 26', '후보 18건 -> 리랭킹(크로스 인코더) 상위 3건', '유사도 임계값 검증 · 기준 0.55 / 최고 0.67']
14:12:51 INFO     httpx2: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

14:12:52 INFO     app.rag.retriever: 정책으로 제외: 구매·계약 규정 - 보안등급(대외비) 차단
14:13:03 INFO     httpx2: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
[2] 법인카드 한도가 얼마인가요?
    run_id RUN-8862 · 근거 3건 · 답변 법인카드 한도는 **법인카드 사용 지침(v2.1)**에 따라 다음과 같습
    단계 ['질의 재작성 · Multi-Query 2건 생성', '벡터 + 키워드 하이브리드 검색 · 벡터 13건 · 키워드 12건 -> 후보 19', '후보 18건 -> 리랭킹(크로스 인코더) 상위 3건', '유사도 임계값 검증 · 기준 0.55 / 최고 0.69']
